# Project 01 (basic) — classifying network traffic: from flows to features

> **Module 15 — ML for Networks** - Format: **Jupyter notebook**
>
> **Why this format?** This is about **understanding data**: looking at real, dirty network
> data, building features, plotting distributions, comparing models. Exactly the exploratory
> workflow a notebook is made for (as in modules 02-04).

## Goal
You work with **real flow data** for the first time (KDD Cup 99, via `scikit-learn`) and:
1. understand the **feature structure** of a flow record (volume, time, protocol, context),
2. build a clean **preprocessing** step (cast types, encode categorical features,
   **log-transform** heavy tails),
3. classify **normal vs. attack** and **the attack types** (13 classes),
4. walk right into the **accuracy trap** - and understand why 99.9 % means nothing.

## Prior knowledge
Module 15 script, sections **1** (flows, features) and **3.1 stage 1** (accuracy trap).
Module 04 (classification, pipelines, metrics), pandas.

## What should work at the end
A random forest with ~99.99 % accuracy - and your insight that the trivial classifier
"everything is normal" already achieves **96.6 %**, so those great numbers say almost nothing.

> **Dataset warning (important, see script 3.6):** KDD Cup 99 is **outdated (1999),
> synthetic, redundant and too easy**. We use it because it delivers real flow features with a
> realistic imbalance and no download hurdle - ideal for learning **methodology**.
> It permits **no** statement about real IDS quality. For real work: UNSW-NB15 / CIC-IDS2017.

> **How to work:** fill in the `# TODO` places yourself. Solution:
> `solution/flow_classification_solution.ipynb`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_kddcup99
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)

RANDOM_STATE = 0
pd.set_option("display.width", 120)
print("pandas", pd.__version__)


## 1 - Loading the data

On its first call `fetch_kddcup99` downloads ~2 MB (cached afterwards in
`~/scikit_learn_data`). We take the subset **`SA`** (the 10 % variant): ~100,000 flows with
**96.6 % normal traffic** - a realistic imbalance.

Every row is a **flow** (one connection) with 41 features. The columns are exactly the ones from
script 1.3: volume (`src_bytes`, `dst_bytes`), time (`duration`), protocol (`protocol_type`,
`service`, `flag`) and **context** (`count`, `srv_count`, ... = connections within the time
window).


In [ ]:
data = fetch_kddcup99(subset="SA", percent10=True, as_frame=True, random_state=RANDOM_STATE)
df = data.frame.copy()
print("Raw shape:", df.shape)
df.head(3)


## 2 - Cleaning up: the first splash of reality

Two typical peculiarities of real data:
1. **All columns are `object`** (even the numbers!) -> they have to be cast.
2. The values are **bytes**: `b'normal.'` instead of `normal`.

> **Warning - a trap almost everybody falls into.** It is tempting to clean the bytes like this:
> ```python
> s.astype(str).str.strip("b'")        # WRONG!
> ```
> `str.strip` removes **all** characters from the set `{b, '}` at both ends - not the prefix
> `b'`. So `b'back.'` becomes **`ack`** (the "b" of *back* is eaten as well!), and the service
> `b'bgp'` becomes `gp`. You silently destroy data and never notice.
> **The correct way** is a real `decode()` - that is what we do below.


In [ ]:
def bytes_to_str(column):
    # Decode bytes properly (NOT with str.strip("b'") - see the warning above!)
    return column.apply(lambda v: v.decode() if isinstance(v, bytes) else str(v))

# Labels: b'normal.' -> normal   (only strip the trailing dot)
df["labels"] = bytes_to_str(df["labels"]).str.rstrip(".")

# separate categorical from numeric columns
CATEGORICAL = ["protocol_type", "service", "flag"]
numeric = [c for c in df.columns if c not in CATEGORICAL + ["labels"]]

for c in CATEGORICAL:
    df[c] = bytes_to_str(df[c])
for c in numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# Cross-check that nothing was destroyed:
assert "back" in set(df["labels"]), "label 'back' broken -> the str.strip trap!"
print("Classes:", df['labels'].nunique())
print(df["labels"].value_counts())
print("\nShare of normal traffic: %.2f %%" % (100 * (df["labels"] == "normal").mean()))
print("Services (excerpt):", sorted(df['service'].unique())[:12])


## 3 - Looking at the data (EDA)

**Your task:** look at the **distribution of `src_bytes`**. Network traffic has **heavy tails**
(script 1.4): a few elephant flows, millions of mice. Plot a histogram of `src_bytes` - once raw,
once **log-transformed** ($\log(1+x)$).


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
# TODO: on the left a histogram of df["src_bytes"] (raw), 50 bins
# TODO: on the right a histogram of np.log1p(df["src_bytes"]), 50 bins
# Set meaningful titles/axis labels.
raise NotImplementedError

print("Range of src_bytes:", df["src_bytes"].min(), "to", df["src_bytes"].max())
print("Median:", df["src_bytes"].median(), "| mean:", round(df["src_bytes"].mean(), 1))


## 4 - Building features

**Your task:** build the design matrix `X` and the binary target `y`:
- `y = 1` if `labels != "normal"` (attack), else 0.
- `X`: all columns except `labels`; encode the **categorical** ones (`protocol_type`, `service`,
  `flag`) with **one-hot** (`pd.get_dummies`); **log-transform** the byte counters.

> **Why the log?** See the plot above: `src_bytes` ranges from 0 to ~10^9. Without the log a
> single elephant flow dominates every distance/gradient computation. Trees hardly care, linear
> models care a lot.


In [ ]:
LOG_COLUMNS = ["src_bytes", "dst_bytes", "duration"]

# TODO: y = (df["labels"] != "normal") as int
y = None

# TODO: X = df without "labels";  transform LOG_COLUMNS with np.log1p;
#       then pd.get_dummies(X, columns=CATEGORICAL); finally .astype(float)
X = None

raise NotImplementedError
print("X:", X.shape, "| share of attacks: %.2f %%" % (100*y.mean()))


## 5 - Splitting and training the models

> **Warning - here we deliberately do something wrong:** we split **randomly**. For network data
> that is **leakage** (script 3.5), because flows of the same attack end up in train *and* test.
> For this introductory project that is fine - but remember: **the numbers below are therefore
> too optimistic.** Project 03 does it properly.

We compare three models:
- **DummyClassifier** (`strategy="most_frequent"`) - always says "normal". The **yardstick**.
- **Logistic regression** - linear, with scaling.
- **Random forest** - an ensemble of trees.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| test:", X_test.shape)

models = {
    "Dummy (always 'normal')": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                            random_state=RANDOM_STATE),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    acc = accuracy_score(y_test, pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_test, pred, average="binary",
                                                    zero_division=0)
    results[name] = dict(Accuracy=acc, Precision=pr, Recall=rc, F1=f1)
    print(f"{name:24s} done.")

pd.DataFrame(results).T.round(5)


## 6 - The accuracy trap

Look at the table closely. The **dummy**, which says "normal" without exception and finds **not a
single attack**, has an accuracy of **96.6 %**.

The random forest has 99.99 % - measured in accuracy that is only **3 percentage points better**
than doing nothing at all. Accuracy **hides** the entire relevant difference, because it is
dominated by the 96.6 % majority class.

Precision/recall/F1 show the truth: dummy = **0.0**, random forest ~ **1.0**.

**The rule to remember:** *under strong class imbalance, accuracy is not a metric but a
delusion.* - And this here is only **stage 1**. Stage 2 (the **base-rate fallacy**) comes in
project 02 and is considerably more unpleasant.


In [ ]:
df_results = pd.DataFrame(results).T
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
df_results["Accuracy"].plot.barh(ax=ax1, color="crimson", xlim=(0.9, 1.005))
ax1.set(title="Accuracy - looks great everywhere (axis starts at 0.9!)", xlabel="accuracy")
df_results[["Precision", "Recall", "F1"]].plot.barh(ax=ax2)
ax2.set(title="Precision/recall/F1 - here you see the difference", xlim=(0, 1.05))
plt.tight_layout(); plt.show()

print("Dummy accuracy        :", round(results["Dummy (always 'normal')"]["Accuracy"], 5))
print("Random forest accuracy:", round(results["Random forest"]["Accuracy"], 5))
print("--> difference in accuracy:",
      round(results["Random forest"]["Accuracy"]-results["Dummy (always 'normal')"]["Accuracy"], 5),
      "  BUT recall: 0.0 vs.",
      round(results["Random forest"]["Recall"], 3))


## 7 - Multi-class: which attack is it?

So far only normal vs. attack. Now the 13 real classes - and in doing so we stumble over a
problem that **real data have all the time**:

Some attack types occur **exactly once** in the whole dataset (`multihop`, `pod`,
`warezmaster`). A *stratified* split is therefore **impossible** - you cannot split a single
example across train *and* test. `train_test_split(..., stratify=...)` raises a `ValueError`
here.

**This is not a bug, it is reality:** in intrusion detection there is always a long tail of
extremely rare classes. You have three options:
1. **remove** the ultra-rare ones (what we do here - honestly documented),
2. **merge** them into a collective class "other attack",
3. treat them as an **anomaly** problem instead of a classification problem (-> **project 03**!).

> Whoever picks option 1 without saying so is prettifying their results: you throw away exactly
> the difficult cases and then report high numbers.


In [ ]:
# How rare are the rarest classes really?
counts = df["labels"].value_counts()
print("The 6 rarest classes:")
print(counts.tail(6))

MIN_PER_CLASS = 10
keep = counts[counts >= MIN_PER_CLASS].index
mask = df["labels"].isin(keep)
print(f"\nRemoved: {(~mask).sum()} flows from {df['labels'].nunique()-len(keep)} "
      f"ultra-rare classes (<{MIN_PER_CLASS} examples) -> {len(keep)} classes remain.")

X_m, y_multi = X[mask.values], df.loc[mask, "labels"]
Xtr, Xte, ytr_m, yte_m = train_test_split(X_m, y_multi, test_size=0.3, stratify=y_multi,
                                          random_state=RANDOM_STATE)
rf_multi = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
rf_multi.fit(Xtr, ytr_m)
pred_m = rf_multi.predict(Xte)

print("\n" + classification_report(yte_m, pred_m, zero_division=0))
print("Read the 'support' column: for classes with 3-5 test examples every")
print("percentage is pure noise - one misclassification = -20 to -33 points.")


## 8 - Which features matter? (and why that is suspicious)

The random forest provides `feature_importances_`. Look at the top 10.


In [ ]:
imp = pd.Series(rf_multi.feature_importances_, index=X.columns).nlargest(10)
plt.figure(figsize=(7, 4))
imp.sort_values().plot.barh(color="teal")
plt.title("Top 10 most important features (random forest)")
plt.xlabel("importance"); plt.tight_layout(); plt.show()
print(imp.round(4))


## 9 - Conclusion

**What you have learned:**
1. **Flow data are real, dirty data** - all columns `object`, labels as bytes, categorical
   features, extreme value ranges.
2. **Heavy tails are the norm** (mean >> median for `src_bytes`) -> **log-transform**.
3. **The accuracy trap**: the dummy reaches 96.6 % without being able to do anything. Under
   imbalance accuracy is worthless - use **precision/recall**.
4. **Rare classes** (`teardrop`: 11 examples) can barely be assessed statistically.

**And now the uncomfortable truth** (script 3.6): the 99.99 % are **too good to be true**. Three
reasons: (a) KDD99 is **redundant and too easy** - artifacts such as `src_bytes` separate the
classes almost perfectly; (b) we split **randomly** -> **leakage**; (c) the base rate of 3.36 %
attacks is **absurdly high** for a real network. In a real network this would look completely
different - **why exactly**, is computed in **project 02**.

### Mini exercises
1. Drop the log transform - what happens to the **logistic regression**
   (and why almost nothing to the random forest)?
2. Train the RF only on `protocol_type`/`service`/`flag` (without the volume features).
   How much accuracy remains?
3. **Remove** `src_bytes` and retrain. Does performance collapse? What does that say about the
   quality of the dataset?
4. Look at the **confusion matrix** (`confusion_matrix`) of the multi-class variant: which
   attacks get confused?

> Reference answers to the mini exercises are at the end of the solution in the folder
> `solution/`.
